In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import logging
import subprocess
from datetime import datetime
import sys
import pytesseract

# --- 1. SETUP LOGGING ---
# A log file is created to record the progress and any errors that occur during the pipeline run.
log_filename = "pipeline.log"
try:
    logging.basicConfig(
        filename=log_filename,
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s"
    )
    logging.info("Starting the full pipeline execution...")
except Exception as e:
    # Fallback to print if logging setup fails
    print(f"ERROR: Could not set up logging to {log_filename}. Check file permissions. Error: {e}")
    sys.exit(1)

# --- 2. DEFINE DIRECTORIES ---

DRIVE_PATH = "/content/drive/MyDrive/Code_Snippet"  # Adjust this path if your folder is elsewhere in Drive

dataset_dir = os.path.join(DRIVE_PATH,"Data Sets")
out_dir = os.path.join(DRIVE_PATH,"Out_put")
ready_dir = os.path.join(DRIVE_PATH,"Ready_dataset")
transformed_ready_dir = os.path.join(DRIVE_PATH,"Ready_Dataset_Transformed_Output")
model_ready_dir = os.path.join(DRIVE_PATH,"Model_Ready_Dataset")
split_dir = os.path.join(DRIVE_PATH,"Split_Dataset")

# In a Colab notebook, __file__ is not defined.
# We will assume scripts are in the DRIVE_PATH or a subdirectory within it.
# For this fix, we'll assume scripts are directly in DRIVE_PATH.
script_dir = DRIVE_PATH


os.makedirs(ready_dir, exist_ok=True)
os.makedirs(out_dir, exist_ok=True)
os.makedirs(split_dir, exist_ok=True)
os.makedirs(model_ready_dir, exist_ok=True)
os.makedirs(transformed_ready_dir, exist_ok=True)
os.makedirs(dataset_dir, exist_ok=True)

# Note on Tesseract OCR:
# In Colab, you typically install Tesseract via apt-get, not by specifying a Windows path.
# Remove or comment out the following line and install tesseract-ocr using:
# !apt-get update && apt-get install -y tesseract-ocr
# pytesseract.pytesseract.tesseract_cmd = r"C:\\Program Files\\Tesseract-OCR\\tesseract.exe"


# --- 3. PIPELINE STAGES ---

# Stage 1: Match Question Papers and Marking Schemes
logging.info("Stage 1: Running QA_Match_Picker.py to find matching PDF pairs.")
print("Running Stage 1...") # Added print for immediate feedback
try:
    match_picker_script_path = os.path.join(script_dir, "QA_Match_Picker.py")
    subprocess.run(["python3", match_picker_script_path, dataset_dir, out_dir], check=True)
    logging.info("Stage 1: Completed successfully. Matched pairs are in the 'Out_put' directory.")
except subprocess.CalledProcessError as e:
    logging.error(f"Stage 1 failed: {e}")
    sys.exit(1)

# Stage 2: Process and Merge Each Matched Pair
logging.info("Stage 2: Processing and merging each matched pair...")
print("Running Stage 2...") # Added print for immediate feedback
matched_pairs_dirs = [d for d in os.listdir(out_dir) if os.path.isdir(os.path.join(out_dir, d))]

if not matched_pairs_dirs:
    logging.warning("No matched directories found in 'Out_put'. Exiting pipeline.")
    sys.exit(0)

for match_dir in matched_pairs_dirs:
    match_path = os.path.join(out_dir, match_dir)
    pdf_files = [f for f in os.listdir(match_path) if f.endswith('.pdf')]
    qp_file = next((f for f in pdf_files if 'QP' in f or 'qp' in f), None)
    ms_file = next((f for f in pdf_files if 'MS' in f or 'ms' in f), None)

    if not qp_file or not ms_file:
        logging.warning(f"Skipping directory '{match_dir}': Missing either QP or MS PDF.")
        continue

    # Process Question Paper
    logging.info(f"Processing question paper: {qp_file}")
    qp_json_path = os.path.join(match_path, "extracted_questions.json")
    try:
        qp_script_path = os.path.join(script_dir, "extracted_questions_to_text.py")
        subprocess.run(["python3", qp_script_path, os.path.join(match_path, qp_file), os.path.join(match_path, "qp_diagrams"), qp_json_path], check=True)
        logging.info("Question paper processed successfully.")
    except subprocess.CalledProcessError as e:
        logging.error(f"Failed to process question paper '{qp_file}': {e}")
        continue

    # Process Marking Scheme
    logging.info(f"Processing marking scheme: {ms_file}")
    ms_json_path = os.path.join(match_path, "extracted_marking_scheme.json")
    try:
        ms_script_path = os.path.join(script_dir, "marking_scheme_to_text_js.py")
        subprocess.run(["python3", ms_script_path, os.path.join(match_path, ms_file), os.path.join(match_path, "ms_diagrams"), ms_json_path], check=True)
        logging.info("Marking scheme processed successfully.")
    except subprocess.CalledProcessError as e:
        logging.error(f"Failed to process marking scheme '{ms_file}': {e}")
        continue

    # Merge QA Pairs
    logging.info(f"Merging data for {match_dir}")
    try:
        merge_script_path = os.path.join(script_dir, "merged_question_marking_scheme_js.py")
        subprocess.run(["python3", merge_script_path, qp_json_path, ms_json_path, ready_dir], check=True)
        logging.info("Merging completed successfully.")
    except subprocess.CalledProcessError as e:
        logging.error(f"Failed to merge data for '{match_dir}': {e}")
        continue

logging.info("Stage 2: All matched pairs processed.")

# Stage 3: Transform JSON Files
logging.info("Stage 3: Running json_file_transformer_updated.py to restructure JSON data.")
print("Running Stage 3...") # Added print for immediate feedback
try:
    transform_script_path = os.path.join(script_dir, "json_file_transformer_updated.py")
    subprocess.run(["python3", transform_script_path, ready_dir, transformed_ready_dir], check=True)
    logging.info("Stage 3: Transformation completed successfully.")
except subprocess.CalledProcessError as e:
    logging.error(f"Stage 3 failed: {e}")
    sys.exit(1)

# Stage 4: Prepare Model-Ready Dataset
logging.info("Stage 4: Running prepare_model_ready_dataset_enhanced.py to prepare data for a model.")
print("Running Stage 4...") # Added print for immediate feedback
try:
    prepare_script_path = os.path.join(script_dir, "prepare_model_ready_dataset_enhanced.py")
    subprocess.run(["python3", prepare_script_path, transformed_ready_dir, model_ready_dir], check=True)
    logging.info("Stage 4: Dataset prepared successfully.")
except subprocess.CalledProcessError as e:
    logging.error(f"Stage 4 failed: {e}")
    sys.exit(1)

# Stage 5: Split the Dataset
logging.info("Stage 5: Running split_clean.py to split the dataset.")
print("Running Stage 5...") # Added print for immediate feedback
try:
    split_script_path = os.path.join(script_dir, "split_clean.py")
    subprocess.run(["python3", split_script_path, model_ready_dir, split_dir], check=True)
    logging.info("Stage 5: Dataset split into train/test/validation sets successfully.")
except subprocess.CalledPulledProcessError as e:
    logging.error(f"Stage 5 failed: {e}")
    sys.exit(1)

logging.info("Full pipeline execution completed successfully.")
print("\n✓ The full pipeline has finished. Check 'pipeline.log' for details.")
print("✓ Final datasets are located in the 'Split_Dataset' directory.")

Running Stage 1...


In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
!apt-get update && apt-get install -y tesseract-ocr

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [80.4 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,002 kB]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Pack

In [ ]:
# Required libraries: pdfplumber, pdf2image, Pillow, pytesseract
# Installation: pip install pdfplumber pdf2image pillow pytesseract

import os
import re
import json
import pdfplumber
from pdf2image import convert_from_path
from PIL import Image
import pytesseract
import sys
import numpy as np
import warnings
import logging # Ensure logging is imported here as well for standalone execution

# Suppress the specific PDF parsing warnings
warnings.filterwarnings("ignore", category=Warning, module="pdfminer")

# If Tesseract isn't on your PATH, you'll need to point pytesseract to it.
# Example for Windows:
# pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
# Example for Linux/macOS, check your installation path.

# --- Configure basic logging for the script ---
# This helps see messages when the script is run standalone via subprocess
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")


# ───────────────────────── Utilities ─────────────────────────
def _dist_vertical(y1, y2):
    """Calculates the absolute vertical distance between two y-coordinates."""
    return abs(y1 - y2)

def _merge_bboxes(bboxes, tolerance=5):
    """
    Merges overlapping or very close bounding boxes.
    This is a critical step for combining fragmented vector elements.
    """
    if not bboxes:
        return []

    # Filter out zero-area or very small bounding boxes
    bboxes = [b for b in bboxes if (b[2] - b[0]) * (b[3] - b[1]) > 1]

    merged = []
    bboxes = sorted(bboxes, key=lambda b: b[1]) # Sort by y-coordinate

    while bboxes:
        current_bbox = list(bboxes.pop(0))
        i = 0
        while i < len(bboxes):
            other_bbox = bboxes[i]
            # Check for vertical and horizontal overlap with a tolerance
            if (max(current_bbox[0], other_bbox[0]) < min(current_bbox[2], other_bbox[2]) + tolerance) and \
               (max(current_bbox[1], other_bbox[1]) < min(current_bbox[3], other_bbox[3]) + tolerance):
                # Merge bounding boxes
                current_bbox[0] = min(current_bbox[0], other_bbox[0])
                current_bbox[1] = min(current_bbox[1], other_bbox[1])
                current_bbox[2] = max(current_bbox[2], other_bbox[2])
                current_bbox[3] = max(current_bbox[3], other_bbox[3])
                bboxes.pop(i)
            else:
                i += 1
        merged.append(current_bbox)
    return merged

def _is_valid_q_text(text):
    """Checks if a string is valid question text (not just numbers, symbols, or headers)."""
    text = text.strip()
    # Reject strings that are just whitespace, numbers, or very short.
    if len(text) < 10 or text.isdigit():
        return False
    # Reject strings that are likely page headers/footers or simple marks.
    if re.fullmatch(r'Page \d+', text, re.IGNORECASE) or re.fullmatch(r'\[\d+]', text) or \
       re.fullmatch(r'\d+', text):
        return False
    return True

# ───────────────────── Question Extraction Logic ─────────────────────
def extract_questions(pdf_path):
    """
    A more robust function to extract questions and their parts from a PDF.
    It uses a location-based approach, which is more reliable for complex layouts.
    Returns a list of questions or an empty list if none are found or an error occurs.
    """
    questions = []
    question_pattern = re.compile(r'^\s*(\d+[a-z]?)(?:$|\s|\s*\()')
    last_q_num = 0

    try:
        with pdfplumber.open(pdf_path) as pdf:
            all_chars = []
            for page_num, page in enumerate(pdf.pages):
                chars = page.chars
                # Filter out tiny characters that are likely noise
                chars = [c for c in chars if c['size'] > 5]
                all_chars.extend(chars)

            if not all_chars:
                logging.warning(f"No readable characters found in {pdf_path}. Skipping question extraction.")
                return []

            # Sort all characters by their vertical position, then horizontal.
            all_chars.sort(key=lambda c: (c['top'], c['x0']))

            current_text = ""
            current_question = None
            current_part = None
            last_y_position = 0

            # Group characters into lines and then into blocks
            line_tolerance = 5
            lines = []
            if all_chars:
                current_line = []
                for char in all_chars:
                    if not current_line or abs(char['top'] - current_line[-1]['top']) < line_tolerance:
                        current_line.append(char)
                    else:
                        lines.append(current_line)
                        current_line = [char]
                if current_line:
                    lines.append(current_line)

            for line in lines:
                line_text = "".join([c['text'] for c in line]).strip()
                # Heuristic to check for a new question. It should start with a number.
                match = question_pattern.match(line_text)
                if match and _is_valid_q_text(line_text):
                    q_id = match.group(1).strip()
                    # Check if it's a new main question or a sub-question
                    q_num_match = re.match(r'^\d+', q_id)
                    if q_num_match:
                        q_num = int(q_num_match.group(0))
                        if q_num > last_q_num or not questions:
                            # This is a new main question.
                            if current_question:
                                current_part['text'] = current_text.strip()
                                current_question['parts'].append(current_part)
                                questions.append(current_question)

                            current_question = {
                                "question_number": q_num,
                                "parts": []
                            }
                            current_part = {
                                "question_id": q_id,
                                "text": line_text[len(q_id):].strip(),
                                "page": line[0]['page_number'],
                                "bbox": [line[0]['x0'], line[0]['top'], line[-1]['x1'], line[-1]['bottom']],
                                "diagrams": [],
                                "marking_scheme": {}
                            }
                            last_q_num = q_num
                            current_text = ""
                        else:
                            # This is a sub-question
                            if current_question and current_part:
                                current_part['text'] = current_text.strip()
                                current_question['parts'].append(current_part)
                                current_part = {
                                    "question_id": q_id,
                                    "text": line_text[len(q_id):].strip(),
                                    "page": line[0]['page_number'],
                                    "bbox": [line[0]['x0'], line[0]['top'], line[-1]['x1'], line[-1]['bottom']],
                                    "diagrams": [],
                                    "marking_scheme": {}
                                }
                                current_text = ""
                elif current_part:
                    current_text += "\n" + line_text

            # Add the last question to the list
            if current_question and current_part:
                current_part['text'] = current_text.strip()
                current_question['parts'].append(current_part)
                questions.append(current_question)

    except Exception as e:
        logging.error(f"Error processing PDF {pdf_path}: {e}")
        # Return empty list on error instead of exiting
        return []

    # Log a warning if no questions were found but don't exit
    if not questions:
        logging.warning(f"No structured questions found in {pdf_path}.")

    return questions

# ───────────────────── Diagram Extraction Logic ─────────────────────
def extract_diagrams(pdf_path, output_dir):
    """Extracts diagrams from a PDF using image extraction and OCR for captioning."""
    diagrams = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                images = page.images
                if images:
                    try:
                        pil_images = convert_from_path(pdf_path, first_page=page_num+1, last_page=page_num+1)
                    except Exception as e:
                         logging.warning(f"Could not convert page {page_num + 1} of {pdf_path} to image for diagram extraction: {e}")
                         continue # Skip to the next page if conversion fails

                    if pil_images:
                        pil_image = pil_images[0]
                        for i, img in enumerate(images):
                            bbox = (img['x0'], img['top'], img['x1'], img['bottom'])
                            # Ensure bbox is valid before cropping
                            if bbox[0] >= bbox[2] or bbox[1] >= bbox[3]:
                                logging.warning(f"Invalid bbox for image on page {page_num + 1}, index {i}: {bbox}. Skipping.")
                                continue

                            try:
                                # Crop the image using its bounding box
                                cropped_img = pil_image.crop(bbox)

                                # Use OCR to get a text caption/descriptor from the image
                                try:
                                    # Use config for better OCR results if needed, e.g., '--psm 6'
                                    caption = pytesseract.image_to_string(cropped_img)
                                    caption = caption.replace('\n', ' ').strip()
                                except Exception as e:
                                    logging.warning(f"OCR failed for image on page {page_num + 1}, index {i}: {e}. Caption will be empty.")
                                    caption = ""

                                # Create a unique filename for the image
                                img_path = os.path.join(output_dir, f"page{page_num+1}_img{i+1}.png")
                                os.makedirs(output_dir, exist_ok=True)
                                cropped_img.save(img_path)

                                diagrams.append({
                                    "id": f"diagram_page{page_num+1}_img{i+1}",
                                    "file_path": img_path,
                                    "caption": caption,
                                    "page": page_num,
                                    "bbox": bbox
                                })
                            except Exception as e:
                                logging.warning(f"Error processing image on page {page_num + 1}, index {i}: {e}. Skipping image.")
                                continue # Skip to the next image if processing fails

    except Exception as e:
         logging.error(f"Error extracting diagrams from {pdf_path}: {e}")
         # Return empty list on error instead of exiting
         return []

    return diagrams

# ───────────────────── Diagram Assignment Logic ─────────────────────
def assign_diagrams_to_questions(questions, diagrams):
    """
    Assigns diagrams to the closest question based on vertical proximity.
    """
    if not diagrams or not questions:
        # Return questions list as is if no diagrams or questions are present
        return questions

    for diagram in diagrams:
        diagram_y = (diagram["bbox"][1] + diagram["bbox"][3]) / 2
        min_dist = float('inf')
        best_match_part = None # Use best_match_part to store the specific part

        for q in questions:
            for part in q["parts"]:
                # Ensure part has a 'bbox' key and it's a list/tuple of 4 numbers
                if 'bbox' in part and isinstance(part['bbox'], (list, tuple)) and len(part['bbox']) == 4:
                    q_y_bottom = part["bbox"][3]
                    dist = _dist_vertical(diagram_y, q_y_bottom)
                    if dist < min_dist:
                        min_dist = dist
                        best_match_part = part # Store the part reference

        if best_match_part:
            # Ensure the part has a 'diagrams' key before appending
            if 'diagrams' not in best_match_part:
                best_match_part['diagrams'] = []
            best_match_part["diagrams"].append(diagram)
        else:
            logging.warning(f"Diagram {diagram['id']} could not be assigned to any question.")

    return questions

# ───────────────────────── Main Function ─────────────────────────
def process_question_paper(pdf_path, diagram_out_dir, json_out_path):
    """Processes a single question paper PDF."""
    logging.info(f"Processing PDF: {pdf_path}")
    os.makedirs(os.path.dirname(json_out_path) or '.', exist_ok=True) # Ensure output directory for JSON exists

    # 1) Extract structured questions and their parts
    logging.info("⋯ extracting structured questions...")
    questions = extract_questions(pdf_path)

    # No exit here, allow subsequent stages to handle empty questions list
    # if not questions:
    #     print("No structured questions found. Exiting.")
    #     sys.exit(1)

    # 2) Extract all diagrams
    logging.info("⋯ extracting diagrams and images…")
    diagrams = extract_diagrams(pdf_path, diagram_out_dir)

    # 3) Assign diagrams to the questions
    logging.info("⋯ assigning diagrams to questions…")
    # Only attempt assignment if both questions and diagrams exist
    if questions and diagrams:
        enriched_questions = assign_diagrams_to_questions(questions, diagrams)
    else:
        enriched_questions = questions # Use the questions list as is if no assignment was possible

    # 4) Save the final JSON
    try:
        with open(json_out_path, "w", encoding="utf-8") as f:
            json.dump(enriched_questions, f, indent=2)
        logging.info(f"✓ Saved extracted data to {json_out_path}")
        print(f"✓ Saved extracted data to {json_out_path}") # Added print for immediate feedback
    except Exception as e:
        logging.error(f"Error saving JSON to {json_out_path}: {e}")
        # Don't exit here, let the main pipeline handle this if necessary
        # sys.exit(1)


if __name__ == "__main__":
    if len(sys.argv) != 4:
        logging.error("Usage: python extracted_questions_to_text.py <path_to_pdf> <diagrams_output_dir> <json_output_path>")
        print("Usage: python extracted_questions_to_text.py <path_to_pdf> <diagrams_output_dir> <json_output_path>") # Added print for immediate feedback
        sys.exit(1)

    pdf_path = sys.argv[1]
    diagram_out_dir = sys.argv[2]
    json_out_path = sys.argv[3]

    process_question_paper(pdf_path, diagram_out_dir, json_out_path)

ModuleNotFoundError: No module named 'pdfplumber'

After installing, you can run the code cell with the error again.

In [ ]:
import os
import shutil
import sys
import logging # Import logging

# Configure logging for this script as well
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")


def normalize_filename(filename, patterns):
    """
    Normalizes a filename by replacing a given pattern with a hyphen.
    Handles multiple potential patterns.
    """
    for pattern in patterns:
        if pattern in filename:
            # Replace the pattern with a hyphen and then split
            return filename.replace(pattern, '-').rsplit('.', 1)[0]
    return None

def collect_all_qp_ms_pairs(input_dir, output_dir):
    """
    Collects and organizes matching Question Paper (QP) and Marking Scheme (MS) files
    from an input directory into structured subdirectories.
    Handles both '-QP-' and '_qp_' naming conventions.
    """
    try:
        os.makedirs(output_dir, exist_ok=True)
        logging.info(f"Ensured output directory exists: {output_dir}")
    except FileExistsError:
         logging.warning(f"Output directory already exists: {output_dir}. Proceeding.")
    except Exception as e:
        logging.error(f"Error creating output directory {output_dir}: {e}")
        sys.exit(1)


    files = os.listdir(input_dir)
    qp_files = [f for f in files if ('-QP-' in f or '_qp_' in f) and f.endswith('.pdf')]
    ms_files = [f for f in files if ('-MS-' in f or '_ms_' in f) and f.endswith('.pdf')]

    # Define the patterns to search for and replace
    qp_patterns = ['-QP-', '_qp_']
    ms_patterns = ['-MS-', '_ms_']

    # Use the helper function to normalize filenames
    qp_bases = {normalize_filename(f, qp_patterns): f for f in qp_files}
    ms_bases = {normalize_filename(f, ms_patterns): f for f in ms_files}

    # Find common base names that exist in both QP and MS dictionaries
    common_bases = sorted(set(qp_bases.keys()) & set(ms_bases.keys()))

    if not common_bases:
        logging.warning("No matching QP-MS PDF pairs found.")
        print("No matching QP-MS PDF pairs found.") # Added print for immediate feedback
        return

    for index, base in enumerate(common_bases, start=1):
        qp_file = qp_bases[base]
        ms_file = ms_bases[base]

        # Create a subdirectory for each matching pair
        sub_dir = os.path.join(output_dir, f"Match_{index:03d}_{base}")
        try:
            os.makedirs(sub_dir, exist_ok=True)
            logging.info(f"Ensured subdirectory exists: {sub_dir}")
        except FileExistsError:
            logging.warning(f"Subdirectory already exists: {sub_dir}. Proceeding.")
        except Exception as e:
            logging.error(f"Error creating subdirectory {sub_dir}: {e}")
            # Decide how to handle this error - either skip the pair or exit
            continue # Skip this pair and try the next one


        # Copy the matched files into the new subdirectory
        try:
            shutil.copy2(os.path.join(input_dir, qp_file), os.path.join(sub_dir, qp_file))
            shutil.copy2(os.path.join(input_dir, ms_file), os.path.join(sub_dir, ms_file))
            logging.info(f"Copied pair '{qp_file}' and '{ms_file}' to '{sub_dir}'.")
            print(f"✓ Copied pair '{qp_file}' and '{ms_file}' to '{sub_dir}'.") # Added print for immediate feedback
        except Exception as e:
            logging.error(f"Error copying files for pair {base} to {sub_dir}: {e}")
            continue # Skip this pair and try the next one


    logging.info(f"All {len(common_bases)} QP-MS pairs have been processed.")
    print(f"\n✓ All {len(common_bases)} QP-MS pairs have been processed.") # Added print for immediate feedback

# Modified to use command-line arguments
if __name__ == "__main__":
    if len(sys.argv) != 3:
        logging.error("Usage: python QA_Match_Picker.py <input_directory> <output_directory>")
        print("Usage: python QA_Match_Picker.py <input_directory> <output_directory>") # Added print for immediate feedback
        sys.exit(1)
    input_dir = sys.argv[1]
    output_dir = sys.argv[2]
    collect_all_qp_ms_pairs(input_dir, output_dir)

FileNotFoundError: [Errno 2] No such file or directory: '-f'

The code for `QA_Match_Picker.py` needs to be saved to a file before the main pipeline script can execute it using `subprocess.run`. Run this cell to save the script.

In [ ]:
# Ensure the script directory is defined (assuming DRIVE_PATH is set in a previous cell)
# If DRIVE_PATH is not set, replace DRIVE_PATH with the actual path.
DRIVE_PATH = "/content/drive/MyDrive/Code_Snippet" # Make sure this matches the path in your pipeline script
script_dir = DRIVE_PATH
os.makedirs(script_dir, exist_ok=True) # Ensure the script directory exists

script_content = """
import os
import shutil
import sys
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

def normalize_filename(filename, patterns):
    for pattern in patterns:
        if pattern in filename:
            return filename.replace(pattern, '-').rsplit('.', 1)[0]
    return None

def collect_all_qp_ms_pairs(input_dir, output_dir):
    try:
        os.makedirs(output_dir, exist_ok=True)
        logging.info(f"Ensured output directory exists: {output_dir}")
    except FileExistsError:
         logging.warning(f"Output directory already exists: {output_dir}. Proceeding.")
    except Exception as e:
        logging.error(f"Error creating output directory {output_dir}: {e}")
        sys.exit(1)

    files = os.listdir(input_dir)
    qp_files = [f for f in files if ('-QP-' in f or '_qp_' in f) and f.endswith('.pdf')]
    ms_files = [f for f in files if ('-MS-' in f or '_ms_' in f) and f.endswith('.pdf')]

    qp_patterns = ['-QP-', '_qp_']
    ms_patterns = ['-MS-', '_ms_']

    qp_bases = {normalize_filename(f, qp_patterns): f for f in qp_files}
    ms_bases = {normalize_filename(f, ms_patterns): f for f in ms_files}

    common_bases = sorted(set(qp_bases.keys()) & set(ms_bases.keys()))

    if not common_bases:
        logging.warning("No matching QP-MS PDF pairs found.")
        print("No matching QP-MS PDF pairs found.")
        return

    for index, base in enumerate(common_bases, start=1):
        qp_file = qp_bases[base]
        ms_file = ms_bases[base]

        sub_dir = os.path.join(output_dir, f"Match_{index:03d}_{base}")
        try:
            os.makedirs(sub_dir, exist_ok=True)
            logging.info(f"Ensured subdirectory exists: {sub_dir}")
        except FileExistsError:
            logging.warning(f"Subdirectory already exists: {sub_dir}. Proceeding.")
        except Exception as e:
            logging.error(f"Error creating subdirectory {sub_dir}: {e}")
            continue

        try:
            shutil.copy2(os.path.join(input_dir, qp_file), os.path.join(sub_dir, qp_file))
            shutil.copy2(os.path.join(input_dir, ms_file), os.path.join(sub_dir, ms_file))
            logging.info(f"Copied pair '{qp_file}' and '{ms_file}' to '{sub_dir}'.")
            print(f"✓ Copied pair '{qp_file}' and '{ms_file}' to '{sub_dir}'.")
        except Exception as e:
            logging.error(f"Error copying files for pair {base} to {sub_dir}: {e}")
            continue

    logging.info(f"All {len(common_bases)} QP-MS pairs have been processed.")
    print(f"✓ All {len(common_bases)} QP-MS pairs have been processed.")

if __name__ == "__main__":
    if len(sys.argv) != 3:
        logging.error("Usage: python QA_Match_Picker.py <input_directory> <output_directory>")
        print("Usage: python QA_Match_Picker.py <input_directory> <output_directory>")
        sys.exit(1)
    input_dir = sys.argv[1]
    output_dir = sys.argv[2]
    collect_all_qp_ms_pairs(input_dir, output_dir)
"""

script_path = os.path.join(script_dir, "QA_Match_Picker.py")

try:
    with open(script_path, "w") as f:
        f.write(script_content)
    print(f"Successfully saved QA_Match_Picker.py to {script_path}")
except Exception as e:
    print(f"Error saving script to {script_path}: {e}")

Successfully saved QA_Match_Picker.py to /content/drive/MyDrive/Code_Snippet/QA_Match_Picker.py


In [ ]:
import os
import json
import re
import logging # Import logging for traceback

# Configure basic logging for the script
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")


def sanitize_json_string(json_str):
    # Remove control characters that can cause JSON parsing issues
    sanitized = re.sub(r'[\x00-\x1F\x7F]', '', json_str)
    return sanitized

def restructure_questions(data):
    # Ensure data is a dictionary and has a 'questions' key
    if not isinstance(data, dict) or "questions" not in data:
        logging.warning("Input data to restructure_questions is not a dictionary or missing 'questions' key.")
        return data # Return original data or handle as appropriate

    question_map = {}
    # Ensure data["questions"] is a list before iterating
    if not isinstance(data["questions"], list):
         logging.warning("data['questions'] is not a list.")
         return data # Return original data or handle as appropriate

    for question in data["questions"]:
        # Ensure each item in questions list is a dictionary
        if not isinstance(question, dict):
            logging.warning(f"Skipping non-dictionary item in questions list: {question}")
            continue # Skip this item

        q_number = question.get("question_number", "")
        # Ensure q_number is a string before checking length
        if not isinstance(q_number, str):
            logging.warning(f"Skipping question with non-string question_number: {q_number}")
            continue

        if len(q_number) == 2:
            question_map[q_number] = {
                "question_number": q_number,
                "parts": question.get("parts", [])
            }
        elif len(q_number) == 3:
            parent_q = q_number[:2]
            if parent_q not in question_map:
                question_map[parent_q] = {
                    "question_number": parent_q,
                    "parts": []
                }
            # Ensure question.get("parts", []) returns a list or is not None before extending
            parts_to_extend = question.get("parts", [])
            # Add explicit check for None before isinstance and extend
            if parts_to_extend is not None and isinstance(parts_to_extend, list):
                 question_map[parent_q]["parts"].extend(parts_to_extend)
            else:
                 # Log a more informative warning if parts are not a list or are None
                 logging.warning(f"Skipping non-list or None parts for question {q_number}: {parts_to_extend}")
        else:
            # Handle other q_number lengths if necessary, or just add the question as is
            question_map[q_number] = question # This might need refinement based on expected structure

    # Convert the map values back to a list for the output
    data["questions"] = list(question_map.values())
    return data

def process_json_folder(input_dir, output_dir):
    try:
        os.makedirs(output_dir, exist_ok=True)
        logging.info(f"Ensured output directory exists: {output_dir}")
    except FileExistsError as e:
         # Explicitly catch FileExistsError, often transient in Colab with subprocess
         logging.warning(f"Caught FileExistsError for {output_dir} with exist_ok=True: {e}. Assuming directory exists and proceeding.")
    except Exception as e:
        # Catch other potential errors during directory creation
        logging.error(f"Error creating output directory {output_dir}: {e}")
        # Decide if this is a fatal error or if processing can continue
        # For now, we'll let the main try-except in __main__ handle exiting
        raise # Re-raise the exception to be caught by the main block


    logging.info(f"Processing JSON files from '{input_dir}' to '{output_dir}'")

    if not os.path.isdir(input_dir):
        logging.error(f"Input directory not found: {input_dir}")
        return # Exit function if input directory doesn't exist

    for filename in os.listdir(input_dir):
        if filename.endswith(".json"):
            input_path = os.path.join(input_dir, filename)
            output_filename = f"transformed_{filename}" # Use the original filename prefix
            output_path = os.path.join(output_dir, output_filename)

            logging.info(f"Processing file: {filename}")
            try:
                with open(input_path, 'r', encoding='utf-8') as f:
                    raw_content = f.read()
                    sanitized_content = sanitize_json_string(raw_content)

                # Try loading JSON
                data = json.loads(sanitized_content)

                # Try transforming data
                transformed_data = restructure_questions(data)

                # Try saving output
                with open(output_path, 'w', encoding='utf-8') as f_out:
                    json.dump(transformed_data, f_out, indent=2)

                logging.info(f"✓ Transformation complete. Output saved to: {output_path}")
                print(f"✓ Transformation complete. Output saved to: {output_path}") # Added print for immediate feedback

            except json.JSONDecodeError as e:
                logging.error(f"JSON decoding error in {filename}: {e}")
                # Log the content that caused the error if possible (be cautious with large content)
                # logging.error(f"Problematic content start: {sanitized_content[:500]}...")
            except Exception as e:
                logging.error(f"Error processing {filename}: {e}", exc_info=True) # Log traceback
                print(f"Error processing {filename}: {e}") # Added print for immediate feedback


if __name__ == "__main__":
    # Check for correct number of command-line arguments
    if len(sys.argv) != 3:
        logging.error("Usage: python json_file_transformer_updated.py <input_directory> <output_directory>")
        print("Usage: python json_file_transformer_updated.py <input_directory> <output_directory>")
        sys.exit(1) # Exit with status 1 if usage is incorrect

    input_dir = sys.argv[1]
    output_dir = sys.argv[2]

    # Add a try-except block around the main processing function call
    try:
        process_json_folder(input_dir, output_dir)
        logging.info("json_file_transformer_updated.py finished successfully.")
    except Exception as e:
        # Catch any unhandled exceptions and log them before exiting
        logging.error(f"An unhandled error occurred during json_file_transformer_updated.py execution: {e}", exc_info=True)
        # Still exit with status 1 to signal failure to the calling process (pipeline)
        sys.exit(1)

ERROR:root:Input directory not found: -f


In [ ]:
import os
import json
from PIL import Image
from transformers import BertTokenizer
from torchvision import transforms
import torch
import logging # Import logging

# Configure basic logging for the script
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")


# Enhanced image augmentation transforms for computer vision tasks.
image_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.GaussianBlur(kernel_size=5),
    transforms.ToTensor()
])

# BERT tokenizer for natural language processing tasks.
# Add a try-except block around tokenizer loading in case of network issues or missing files
try:
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
except Exception as e:
    logging.error(f"Error loading BERT tokenizer: {e}", exc_info=True)
    # Decide how to handle failure to load tokenizer - exit or use a dummy?
    # For now, we'll let the script proceed, but text processing will fail.
    tokenizer = None # Set tokenizer to None if loading fails


def process_entry(entry):
    """
    Processes a single data entry (a dictionary) from the input JSON file.
    It tokenizes the text and handles image data.

    Args:
        entry (dict): A dictionary representing a single data entry.

    Returns:
        dict: The processed dictionary with tokenized text and image data, or None if the entry is invalid or processing fails.
    """
    processed = {}

    # Process the text field. The key 'body' is used based on your sample JSON.
    text = entry.get("body")
    # Add check if text is None or not a string before processing
    if text is not None and isinstance(text, str):
        cleaned_text = text.strip()
        # Add try-except around tokenization
        if tokenizer: # Only attempt tokenization if tokenizer was loaded successfully
            try:
                # Tokenize the text and convert it to a PyTorch tensor.
                # Added explicit None check for cleaned_text although isinstance(text, str) should handle this
                tokenized = tokenizer(cleaned_text if cleaned_text is not None else "", return_tensors="pt", truncation=True, padding="max_length", max_length=512) # Added padding and max_length
                # Squeeze the tensor to remove the batch dimension and convert to a list for JSON serialization.
                # Ensure tokenized is not None before accessing input_ids
                if tokenized and "input_ids" in tokenized:
                    processed["text"] = tokenized["input_ids"].squeeze().tolist()
                else:
                    logging.warning(f"Tokenization failed for entry: {entry}. Tokenizer output unexpected.")

            except Exception as e:
                logging.error(f"Error during text tokenization for entry: {entry}. Error: {e}", exc_info=True)
                # Continue processing other parts of the entry even if text tokenization fails
        else:
             logging.warning(f"Tokenizer not loaded. Skipping text tokenization for entry: {entry}")

    # Process the 'marking_scheme' field as a dictionary.
    marking_scheme = entry.get("marking_scheme", {})
    # Ensure marking_scheme is a dictionary, default={} should handle missing key, but check explicitly if value is None
    if marking_scheme is not None and isinstance(marking_scheme, dict):
        processed["marking_scheme"] = marking_scheme
    elif marking_scheme is not None: # It exists but is not a dictionary
         logging.warning(f"Marking scheme for entry is not a dictionary: {entry}. Value: {marking_scheme}")


    # Correctly process the 'diagrams' field.
    # The image path is nested, so we get it carefully from the 'diagrams' list.
    image_file = None
    diagrams_list = entry.get("diagrams")
    # Add explicit checks for diagrams_list being a list and not empty
    if diagrams_list is not None and isinstance(diagrams_list, list) and len(diagrams_list) > 0:
        # Assuming the first item in the diagrams list contains the image_file.
        # Add try-except around accessing list elements and .get()
        try:
            if isinstance(diagrams_list[0], dict):
                image_file = diagrams_list[0].get("image_file")
            else:
                 logging.warning(f"First item in diagrams list is not a dictionary for entry: {entry}. Value: {diagrams_list[0]}")
        except IndexError:
             logging.warning(f"Diagrams list is empty or index 0 is invalid for entry: {entry}. List: {diagrams_list}")
        except Exception as e:
             logging.error(f"Error accessing first diagram item for entry: {entry}. Error: {e}", exc_info=True)


        # Add the diagrams list to the processed dictionary for completeness.
        processed["diagrams"] = diagrams_list
    elif diagrams_list is not None: # It exists but is not a list
         logging.warning(f"Diagrams field for entry is not a list: {entry}. Value: {diagrams_list}")


    # Process image path if it exists and the file exists on the system.
    # The Image.open try-except is already here, good.
    if image_file and isinstance(image_file, str) and os.path.exists(image_file):
        try:
            image = Image.open(image_file).convert("RGB")
            # Apply image transformations and convert to a PyTorch tensor.
            augmented = image_transforms(image)
            # Convert the tensor to a list for JSON serialization.
            processed["image"] = augmented.tolist()
        except Exception as e:
            logging.error(f"Error processing image {image_file} for entry: {entry}. Error: {e}", exc_info=True)


    # Ensure a valid entry is returned.
    # Only return processed if it contains at least one key (text, marking_scheme, image, diagrams)
    if processed:
        return processed
    return None # Return None if no data could be processed for this entry


def process_files_in_folder(input_folder, output_folder):
    """
    Main function to orchestrate the data processing pipeline.
    It iterates through all JSON files in the input folder, processes them,
    and saves the output to the specified folder.

    Args:
        input_folder (str): The path to the folder containing input JSON files.
        output_folder (str): The path to the folder where processed files will be saved.
    """
    # Create the output directory if it doesn't exist.
    # Add try-except around directory creation as seen needed in previous script
    try:
        os.makedirs(output_folder, exist_ok=True)
        logging.info(f"Ensured output directory exists: {output_folder}")
    except FileExistsError as e:
         logging.warning(f"Caught FileExistsError for {output_folder} with exist_ok=True: {e}. Assuming directory exists and proceeding.")
    except Exception as e:
        logging.error(f"Error creating output directory {output_folder}: {e}", exc_info=True)
        # If output directory cannot be created, cannot proceed, so log and return/exit
        return # Or sys.exit(1) if this is a fatal error for the script

    # Get a list of all files in the input directory.
    if not os.path.isdir(input_folder):
        logging.error(f"Input directory not found: {input_folder}")
        return

    for fname in os.listdir(input_folder):
        if fname.endswith(".json"):
            input_path = os.path.join(input_folder, fname)
            output_path = os.path.join(output_folder, fname)

            # Add try-except around processing each file
            try:
                logging.info(f"Processing file: {fname}")
                # Open and load the JSON data from the input file.
                with open(input_path, "r", encoding="utf-8") as f_in:
                    data = json.load(f_in)

                # The JSON structure has a 'questions' key with a list of questions.
                # We need to iterate through these and their 'parts'.
                processed_entries = []
                # Ensure data.get("questions", []) is iterable and handle potential issues
                questions_list = data.get("questions", [])
                if questions_list is not None and isinstance(questions_list, list):
                    for question in questions_list:
                         # Ensure question is a dictionary
                         if question is not None and isinstance(question, dict):
                            parts_list = question.get("parts", [])
                            # Ensure parts_list is iterable
                            if parts_list is not None and isinstance(parts_list, list):
                                for part in parts_list:
                                     # Ensure part is a dictionary before processing
                                     if part is not None and isinstance(part, dict):
                                        # Add try-except around processing each entry/part
                                        try:
                                            processed_entry = process_entry(part)
                                            if processed_entry:  # Check if the entry was processed successfully
                                                processed_entries.append(processed_entry)
                                        except Exception as e:
                                             logging.error(f"Error processing entry/part for file {fname}: {part}. Error: {e}", exc_info=True)
                                             # Continue to the next part even if one fails
                                     elif part is not None:
                                          logging.warning(f"Item in parts list is not a dictionary for file {fname}: {part}. Value: {part}")
                            elif parts_list is not None:
                                 logging.warning(f"Parts list for a question is not a list for file {fname}: {parts_list}. Value: {parts_list}")
                         elif question is not None:
                              logging.warning(f"Item in questions list is not a dictionary for file {fname}: {question}. Value: {question}")
                elif questions_list is not None:
                     logging.warning(f"Questions field in data is not a list for file {fname}: {questions_list}. Value: {questions_list}")


                # Save the processed list of entries to a new file.
                # Only save if there are processed entries
                if processed_entries:
                    try:
                        with open(output_path, "w", encoding="utf-8") as f_out:
                            json.dump(processed_entries, f_out, indent=2)
                        logging.info(f"✓ Processed and saved: {fname}")
                        print(f"✓ Processed and saved: {fname}")
                    except Exception as e:
                         logging.error(f"Error saving processed data for file {fname} to {output_path}: {e}", exc_info=True)
                         print(f"Error saving processed data for file {fname}: {e}") # Added print for immediate feedback
                else:
                     logging.warning(f"No valid entries processed for file {fname}. Not saving output.")
                     print(f"No valid entries processed for file {fname}. Not saving output.") # Added print for immediate feedback

            except json.JSONDecodeError as e:
                logging.error(f"JSON decoding error in file {fname}: {e}", exc_info=True)
                print(f"JSON decoding error in file {fname}: {e}") # Added print for immediate feedback
            except Exception as e:
                logging.error(f"Error processing file {fname}: {e}", exc_info=True)
                print(f"Error processing file {fname}: {e}") # Added print for immediate feedback


# --- Main Execution ---
if __name__ == "__main__":
    # Ensure correct number of command-line arguments (input folder, output folder)
    if len(sys.argv) != 3:
        logging.error("Usage: python prepare_model_ready_dataset_enhanced.py <input_folder> <output_folder>")
        print("Usage: python prepare_model_ready_dataset_enhanced.py <input_folder> <output_folder>")
        sys.exit(1)

    input_folder = sys.argv[1]
    output_folder = sys.argv[2]

    # Add try-except around the main call to process_files_in_folder
    try:
        logging.info("Starting prepare_model_ready_dataset_enhanced.py execution...")
        process_files_in_folder(input_folder, output_folder)
        logging.info("prepare_model_ready_dataset_enhanced.py finished.")
        print("\n✓ All data processing attempted.")
    except Exception as e:
        logging.error(f"An unhandled error occurred during script execution: {e}", exc_info=True)
        sys.exit(1)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

ERROR:root:Input directory not found: -f



✓ All data processing attempted.


In [ ]:
import os
import json
from PIL import Image
from transformers import BertTokenizer
from torchvision import transforms
import torch
import random
import logging # Import logging

# Configure basic logging for the script
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")


# Enhanced image augmentation transforms for computer vision tasks.
# These might not be strictly necessary for splitting, but keeping them as they were in the original code.
image_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.GaussianBlur(kernel_size=5),
    transforms.ToTensor()
])

# BERT tokenizer for natural language processing tasks.
# Also potentially not needed for just splitting, but keeping as in original code.
try:
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
except Exception as e:
    logging.warning(f"Could not load BERT tokenizer: {e}. Text tokenization will not be available.")
    tokenizer = None


def process_and_split_data(input_folder, output_root):
    """
    Processes all JSON files in the input folder and splits the data into
    train, test, validation, and test_labels directories.

    Args:
        input_folder (str): The path to the folder containing input JSON files.
        output_root (str): The root directory where the new split folders will be created.
    """
    logging.info(f"Starting data splitting from '{input_folder}' to '{output_root}'")
    # Define the output folders based on the output root.
    train_folder = os.path.join(output_root, "train")
    test_folder = os.path.join(output_root, "test")
    validation_folder = os.path.join(output_root, "validation")
    test_labels_folder = os.path.join(output_root, "test_labels")

    # Create all output directories if they don't exist.
    # Add try-except around directory creation as seen needed in previous script
    try:
        for folder in [train_folder, test_folder, validation_folder, test_labels_folder]:
            try:
                os.makedirs(folder, exist_ok=True)
                logging.info(f"Ensured output directory exists: {folder}")
            except NotADirectoryError as e:
                 logging.warning(f"Caught NotADirectoryError for {folder} during creation: {e}. Skipping creation for this path.")
            except FileExistsError as e:
                 logging.warning(f"Caught FileExistsError for {folder} with exist_ok=True: {e}. Assuming directory exists and proceeding.")
            except Exception as e:
                 # Catch other potential errors during directory creation
                 logging.error(f"Error creating output directory {folder}: {e}", exc_info=True)


    except Exception as e:
         logging.error(f"An unexpected error occurred during output directory setup: {e}", exc_info=True)
         # If output directories cannot be created, cannot proceed, return/exit
         return # Or sys.exit(1) if this is a fatal error for the script


    # Get a list of all files in the input directory.
    if not os.path.isdir(input_folder):
        logging.error(f"Input directory not found: {input_folder}")
        return

    for fname in os.listdir(input_folder):
        if fname.endswith(".json"):
            input_path = os.path.join(input_folder, fname)
            logging.info(f"Processing file for splitting: {fname}")

            try:
                with open(input_path, "r", encoding="utf-8") as f_in:
                    data = json.load(f_in)

                # Add check if data is None or not a list before processing
                if data is None or not isinstance(data, list):
                    logging.warning(f"Skipping file {fname}: Data is empty or not a list. Value: {data}")
                    print(f"Skipping file {fname}: Data is empty or not a list.") # Added print for immediate feedback
                    continue # Skip to the next file


                processed_entries = []
                # Assuming the input JSON is a list of entries/dictionaries
                # The previous stages should output a list of processed question parts.
                # Iterate through the loaded data.
                for entry in data:
                     # Optional: Add a check here if 'entry' is a dictionary if needed
                     # if isinstance(entry, dict):
                     processed_entries.append(entry)
                     # else:
                     #     logging.warning(f"Skipping invalid entry in file {fname}: {entry}. Not a dictionary.")


                if not processed_entries:
                     logging.warning(f"No valid entries found in file {fname}. Skipping splitting for this file.")
                     print(f"No valid entries found in file {fname}. Skipping splitting.") # Added print for immediate feedback
                     continue # Skip to the next file if no entries were processed


                # Shuffle the data to ensure a random distribution.
                random.shuffle(processed_entries)

                # Calculate the split indices.
                total_entries = len(processed_entries)
                train_split = int(total_entries * train_ratio)
                validation_split = train_split + int(total_entries * validation_ratio)

                # Ensure split indices are within bounds
                train_split = max(0, min(train_split, total_entries))
                validation_split = max(train_split, min(validation_split, total_entries))

                # Split the data into respective lists.
                train_data = processed_entries[:train_split]
                validation_data = processed_entries[train_split:validation_split]
                test_data = processed_entries[validation_split:]

                # Extract only the marking schemes for the test labels.
                test_labels_data = []
                for item in test_data:
                    label_entry = {}
                    # Add checks to ensure 'item' is a dictionary before accessing keys
                    if isinstance(item, dict):
                        if "marking_scheme" in item:
                            label_entry["marking_scheme"] = item["marking_scheme"]
                        if "diagrams" in item:
                            label_entry["diagrams"] = item["diagrams"]
                        # Optional: Add a unique ID or identifier if available and useful for labels
                        if "question_id" in item: # Assuming 'question_id' might be in the processed entry
                             label_entry["question_id"] = item["question_id"]
                        test_labels_data.append(label_entry)
                    else:
                        logging.warning(f"Skipping non-dictionary item in test_data for file {fname}: {item}. Cannot extract labels.")


                # Save the split data to the respective folders.
                # Add try-except around saving files
                try:
                    if train_data: # Only save if there is data
                        with open(os.path.join(train_folder, fname), "w", encoding="utf-8") as f_out:
                            json.dump(train_data, f_out, indent=2)
                        logging.info(f"Saved train data for {fname}")

                    if validation_data: # Only save if there is data
                        with open(os.path.join(validation_folder, fname), "w", encoding="utf-8") as f_out:
                            json.dump(validation_data, f_out, indent=2)
                        logging.info(f"Saved validation data for {fname}")

                    if test_data: # Only save if there is data
                        with open(os.path.join(test_folder, fname), "w", encoding="utf-8") as f_out:
                            # Remove marking scheme from test data before saving.
                            # This is crucial for a real-world test set.
                            test_data_no_labels = [
                                # Add check if d is a dictionary before trying to access items
                                {k: v for k, v in d.items() if k != 'marking_scheme'} if isinstance(d, dict) else d # Keep non-dict items if any, or filter
                                for d in test_data
                            ]
                            json.dump(test_data_no_labels, f_out, indent=2)
                        logging.info(f"Saved test data (without labels) for {fname}")


                    if test_labels_data: # Only save if there are labels
                        # Save test labels data (includes marking scheme)
                        test_labels_output_path = os.path.join(test_labels_folder, f"labels_{fname}") # Use a distinct filename for labels
                        with open(test_labels_output_path, "w", encoding="utf-8") as f_out:
                            json.dump(test_labels_data, f_out, indent=2)
                        logging.info(f"Saved test labels for {fname}")

                    print(f"✓ Processed and split: {fname}") # Added print for immediate feedback

                except Exception as e:
                     logging.error(f"Error saving split data for file {fname}: {e}", exc_info=True)
                     print(f"Error saving split data for file {fname}: {e}") # Added print for immediate feedback


            except json.JSONDecodeError as e:
                logging.error(f"JSON decoding error in file {fname}: {e}", exc_info=True)
                print(f"JSON decoding error in file {fname}: {e}") # Added print for immediate feedback
            except Exception as e:
                logging.error(f"Error processing file {fname}: {e}", exc_info=True)
                print(f"Error processing file {fname}: {e}") # Added print for immediate feedback


# --- Main Execution ---
if __name__ == "__main__":
    # Ensure correct number of command-line arguments (input folder, output folder)
    if len(sys.argv) != 3:
        logging.error("Usage: python split_clean.py <input_folder> <output_folder>")
        print("Usage: python split_clean.py <input_folder> <output_folder>")
        sys.exit(1)

    input_folder = sys.argv[1]
    output_root = sys.argv[2] # Renamed to output_root to match function


    # Add try-except around the main call to process_and_split_data
    try:
        logging.info("Starting split_clean.py execution...")
        process_and_split_data(input_folder, output_root)
        logging.info("split_clean.py finished.")
        print("\n✓ All data split and saved to Split_Datasets.") # Print completion message
    except Exception as e:
        logging.error(f"An unhandled error occurred during script execution: {e}", exc_info=True)
        sys.exit(1)

ERROR:root:Input directory not found: -f



✓ All data split and saved to Split_Datasets.


The code for `split_clean.py` needs to be saved to a file before the main pipeline script can execute it using `subprocess.run`. Run this cell to save the script.

In [ ]:
# Ensure the script directory is defined (assuming DRIVE_PATH is set in a previous cell)
# If DRIVE_PATH is not set, replace DRIVE_PATH with the actual path.
DRIVE_PATH = "/content/drive/MyDrive/Code_Snippet" # Make sure this matches the path in your pipeline script
script_dir = DRIVE_PATH
os.makedirs(script_dir, exist_ok=True) # Ensure the script directory exists

script_content = """
import os
import json
from PIL import Image
from transformers import BertTokenizer
from torchvision import transforms
import torch
import random
import logging # Import logging

# Configure basic logging for the script
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")


# Enhanced image augmentation transforms for computer vision tasks.
# These might not be strictly necessary for splitting, but keeping them as they were in the original code.
image_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.GaussianBlur(kernel_size=5),
    transforms.ToTensor()
])

# BERT tokenizer for natural language processing tasks.
# Also potentially not needed for just splitting, but keeping as in original code.
try:
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
except Exception as e:
    logging.warning(f"Could not load BERT tokenizer: {e}. Text tokenization will not be available.")
    tokenizer = None


def process_and_split_data(input_folder, output_root):
    \"\"\"
    Processes all JSON files in the input folder and splits the data into
    train, test, validation, and test_labels directories.

    Args:
        input_folder (str): The path to the folder containing input JSON files.
        output_root (str): The root directory where the new split folders will be created.
    \"\"\"
    logging.info(f"Starting data splitting from '{input_folder}' to '{output_root}'")
    # Define the output folders based on the output root.
    train_folder = os.path.join(output_root, "train")
    test_folder = os.path.join(output_root, "test")
    validation_folder = os.path.join(output_root, "validation")
    test_labels_folder = os.path.join(output_root, "test_labels")

    # Create all output directories if they don't exist.
    try:
        for folder in [train_folder, test_folder, validation_folder, test_labels_folder]:
            try:
                os.makedirs(folder, exist_ok=True)
                logging.info(f"Ensured output directory exists: {folder}")
            except NotADirectoryError as e:
                 logging.warning(f"Caught NotADirectoryError for {folder} during creation: {e}. Skipping creation for this path.")
            except FileExistsError as e:
                 logging.warning(f"Caught FileExistsError for {folder} with exist_ok=True: {e}. Assuming directory exists and proceeding.")
            except Exception as e:
                 # Catch other potential errors during directory creation
                 logging.error(f"Error creating output directory {folder}: {e}", exc_info=True)


    except Exception as e:
         logging.error(f"An unexpected error occurred during output directory setup: {e}", exc_info=True)
         # If output directories cannot be created, cannot proceed, return/exit
         return # Or sys.exit(1) if this is a fatal error for the script


    # Define the split ratios for the dataset.
    train_ratio = 0.70
    validation_ratio = 0.10
    test_ratio = 0.20

    # Get a list of all files in the input directory.
    if not os.path.isdir(input_folder):
        logging.error(f"Input directory not found: {input_folder}")
        return

    for fname in os.listdir(input_folder):
        if fname.endswith(".json"):
            input_path = os.path.join(input_folder, fname)
            logging.info(f"Processing file for splitting: {fname}")

            try:
                with open(input_path, "r", encoding="utf-8") as f_in:
                    data = json.load(f_in)

                # Add check if data is None or not a list before processing
                if data is None or not isinstance(data, list):
                    logging.warning(f"Skipping file {fname}: Data is empty or not a list. Value: {data}")
                    print(f"Skipping file {fname}: Data is empty or not a list.") # Added print for immediate feedback
                    continue # Skip to the next file


                processed_entries = []
                # Assuming the input JSON is a list of entries/dictionaries
                # The previous stages should output a list of processed question parts.
                # Iterate through the loaded data.
                for entry in data:
                     # Optional: Add a check here if 'entry' is a dictionary if needed
                     # if isinstance(entry, dict):
                     processed_entries.append(entry)
                     # else:
                     #     logging.warning(f"Skipping invalid entry in file {fname}: {entry}. Not a dictionary.")


                if not processed_entries:
                     logging.warning(f"No valid entries found in file {fname}. Skipping splitting for this file.")
                     print(f"No valid entries found in file {fname}. Skipping splitting.") # Added print for immediate feedback
                     continue # Skip to the next file if no entries were processed


                # Shuffle the data to ensure a random distribution.
                random.shuffle(processed_entries)

                # Calculate the split indices.
                total_entries = len(processed_entries)
                train_split = int(total_entries * train_ratio)
                validation_split = train_split + int(total_entries * validation_ratio)

                # Ensure split indices are within bounds
                train_split = max(0, min(train_split, total_entries))
                validation_split = max(train_split, min(validation_split, total_entries))

                # Split the data into respective lists.
                train_data = processed_entries[:train_split]
                validation_data = processed_entries[train_split:validation_split]
                test_data = processed_entries[validation_split:]

                # Extract only the marking schemes for the test labels.
                test_labels_data = []
                for item in test_data:
                    label_entry = {}
                    # Add checks to ensure 'item' is a dictionary before accessing keys
                    if isinstance(item, dict):
                        if "marking_scheme" in item:
                            label_entry["marking_scheme"] = item["marking_scheme"]
                        if "diagrams" in item:
                            label_entry["diagrams"] = item["diagrams"]
                        # Optional: Add a unique ID or identifier if available and useful for labels
                        if "question_id" in item: # Assuming 'question_id' might be in the processed entry
                             label_entry["question_id"] = item["question_id"]
                        test_labels_data.append(label_entry)
                    else:
                        logging.warning(f"Skipping non-dictionary item in test_data for file {fname}: {item}. Cannot extract labels.")


                # Save the split data to the respective folders.
                # Add try-except around saving files
                try:
                    if train_data: # Only save if there is data
                        with open(os.path.join(train_folder, fname), "w", encoding="utf-8") as f_out:
                            json.dump(train_data, f_out, indent=2)
                        logging.info(f"Saved train data for {fname}")

                    if validation_data: # Only save if there is data
                        with open(os.path.join(validation_folder, fname), "w", encoding="utf-8") as f_out:
                            json.dump(validation_data, f_out, indent=2)
                        logging.info(f"Saved validation data for {fname}")

                    if test_data: # Only save if there is data
                        with open(os.path.join(test_folder, fname), "w", encoding="utf-8") as f_out:
                            # Remove marking scheme from test data before saving.
                            # This is crucial for a real-world test set.
                            test_data_no_labels = [
                                # Add check if d is a dictionary before trying to access items
                                {k: v for k, v in d.items() if k != 'marking_scheme'} if isinstance(d, dict) else d # Keep non-dict items if any, or filter
                                for d in test_data
                            ]
                            json.dump(test_data_no_labels, f_out, indent=2)
                        logging.info(f"Saved test data (without labels) for {fname}")


                    if test_labels_data: # Only save if there are labels
                        # Save test labels data (includes marking scheme)
                        test_labels_output_path = os.path.join(test_labels_folder, f"labels_{fname}") # Use a distinct filename for labels
                        with open(test_labels_output_path, "w", encoding="utf-8") as f_out:
                            json.dump(test_labels_data, f_out, indent=2)
                        logging.info(f"Saved test labels for {fname}")

                    print(f"✓ Processed and split: {fname}") # Added print for immediate feedback

                except Exception as e:
                     logging.error(f"Error saving split data for file {fname}: {e}", exc_info=True)
                     print(f"Error saving split data for file {fname}: {e}") # Added print for immediate feedback


            except json.JSONDecodeError as e:
                logging.error(f"JSON decoding error in file {fname}: {e}", exc_info=True)
                print(f"JSON decoding error in file {fname}: {e}") # Added print for immediate feedback
            except Exception as e:
                logging.error(f"Error processing file {fname}: {e}", exc_info=True)
                print(f"Error processing file {fname}: {e}") # Added print for immediate feedback


# --- Main Execution ---
if __name__ == "__main__":
    # Ensure correct number of command-line arguments (input folder, output folder)
    if len(sys.argv) != 3:
        logging.error("Usage: python split_clean.py <input_folder> <output_folder>")
        print("Usage: python split_clean.py <input_folder> <output_folder>")
        sys.exit(1)

    input_folder = sys.argv[1]
    output_root = sys.argv[2] # Renamed to output_root to match function


    # Add try-except around the main call to process_and_split_data
    try:
        logging.info("Starting split_clean.py execution...")
        process_and_split_data(input_folder, output_root)
        logging.info("split_clean.py finished.")
        print("\n✓ All data split and saved to Split_Datasets.") # Print completion message
    except Exception as e:
        logging.error(f"An unhandled error occurred during script execution: {e}", exc_info=True)
        sys.exit(1)
""" # Added missing closing triple quotes

script_path = os.path.join(script_dir, "split_clean.py")

try:
    with open(script_path, "w") as f:
        f.write(script_content)
    print(f"Successfully saved split_clean.py to {script_path}")
except Exception as e:
    print(f"Error saving script to {script_path}: {e}")

Successfully saved split_clean.py to /content/drive/MyDrive/Code_Snippet/split_clean.py


In [ ]:
# Required libraries: pdfplumber, pdf2image, Pillow, pytesseract
# Installation: pip install pdfplumber pdf2image pillow pytesseract

import os
import re
import json
import pdfplumber
from pdf2image import convert_from_path
from PIL import Image
import pytesseract
import sys
import warnings
import logging # Import logging

# Suppress the specific PDF parsing warnings
warnings.filterwarnings("ignore", category=Warning, module="pdfminer")

# If Tesseract isn't on your PATH, you'll need to point pytesseract to it.
# Example for Windows:
# pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
# Example for Linux/macOS, check your installation path.

# --- Configure basic logging for the script ---
# This helps see messages when the script is run standalone via subprocess
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")


# ───────────────────────── Utilities ─────────────────────────
def _dist_vertical(y1, y2):
    """Calculates the absolute vertical distance between two y-coordinates."""
    return abs(y1 - y2)

def _merge_bboxes(bboxes, tolerance=5):
    """
    Merges overlapping or very close bounding boxes.
    This is a critical step for combining fragmented vector elements.
    """
    if not bboxes:
        return []

    bboxes = [b for b in bboxes if (b[2] - b[0]) * (b[3] - b[1]) > 1]

    merged = []
    bboxes = sorted(bboxes, key=lambda b: b[1])

    while bboxes:
        current = list(bboxes.pop(0))
        i = 0
        while i < len(bboxes):
            next_box = bboxes[i]
            if (max(current[0], next_box[0]) < min(current[2], next_box[2]) + tolerance and # Added tolerance here
                max(current[1], next_box[1]) < min(current[3], next_box[3]) + tolerance): # Added tolerance here
                current[0] = min(current[0], next_box[0])
                current[1] = min(current[1], next_box[1])
                current[2] = max(current[2], next_box[2])
                current[3] = max(current[3], next_box[3])
                bboxes.pop(i)
            else:
                i += 1
        merged.append(tuple(current))
    return merged

def extract_diagrams(pdf_path, out_dir):
    """Extracts diagrams/images from a PDF and saves them as PNGs."""
    logging.info(f"Extracting images from {pdf_path}...") # Changed print to logging
    os.makedirs(out_dir, exist_ok=True)
    images_data = []

    try:
        pages = convert_from_path(pdf_path)
    except Exception as e:
        logging.error(f"ERROR: Could not convert PDF to images. Please check your poppler installation. Error: {e}", exc_info=True) # Changed print to logging
        return images_data

    for page_num, page in enumerate(pages):
        filename = f"page_{page_num+1}.png"
        filepath = os.path.join(out_dir, filename)
        # Add try-except around saving pages
        try:
            page.save(filepath, "PNG")
        except Exception as e:
            logging.warning(f"Could not save page {page_num + 1} as image to {filepath}: {e}", exc_info=True)
            continue # Skip to the next page if saving fails


        with pdfplumber.open(pdf_path) as pdf:
            # Ensure pdf_page exists before accessing images
            if page_num < len(pdf.pages):
                pdf_page = pdf.pages[page_num]
                images_on_page = pdf_page.images

                for img_info in images_on_page:
                    x0, top, x1, bottom = img_info['x0'], img_info['top'], img_info['x1'], img_info['bottom']
                    diagram_id = f"diagram_{page_num+1}_{len(images_data)+1}"
                    images_data.append({
                        "diagram_id": diagram_id,
                        "page_number": page_num + 1,
                        "bbox": (x0, top, x1, bottom),
                        "path": filepath # This path is for the whole page image, not the cropped diagram
                    })
            else:
                 logging.warning(f"Page {page_num + 1} not found in pdfplumber object for {pdf_path}.")

    return images_data # Note: This only extracts full page images and their bounding boxes, not cropped diagrams. OCR isn't used here.


# ───────────────────────── Main Logic for Marking Scheme ─────────────────────────

def extract_marking_scheme_data(pdf_path):
    """
    Extracts structured marking scheme data from a PDF by detecting and
    parsing the main question/answer table.
    """
    marking_schemes = []

    try: # Add try-except around PDF opening and text extraction
        with pdfplumber.open(pdf_path) as pdf:
            text = "".join(page.extract_text(x_tolerance=1) for page in pdf.pages)
            if not text:
                logging.warning(f"No readable text found in {pdf_path}. Skipping marking scheme extraction.") # Changed print to logging
                return []
    except Exception as e:
        logging.error(f"Error opening or extracting text from {pdf_path}: {e}", exc_info=True)
        return [] # Return empty list on error


    # Regex to capture the full question part (e.g., 1(a)(i)) and the content that follows
    # Made pattern more flexible for spacing
    ms_pattern = re.compile(
        r'^\s*(\d+\s*\(?[a-z]\s*\)?)(?:\s*\(?[ivx]+\s*\)?)?\s*(.*)', # Adjusted regex
        re.MULTILINE | re.DOTALL
    )

    matches = list(ms_pattern.finditer(text))

    for i, m in enumerate(matches):
        full_id_raw = m.group(1).strip()
        full_id = re.sub(r'\s+', '', full_id_raw)

        content = m.group(2).strip()

        # The rest of the content is from the end of this match to the beginning of the next one
        if i + 1 < len(matches):
            next_start = matches[i+1].start()
            content_from_pdf = text[m.start():next_start].strip()
        else:
            content_from_pdf = text[m.start():].strip()

        # Extract just the content text from the full entry
        # Split by lines, skip the first line (which has the ID), rejoin, and remove lines that are just numbers
        content_lines = content_from_pdf.split('\n')
        if len(content_lines) > 1:
             content = '\n'.join(line for line in content_lines[1:] if not re.fullmatch(r'\s*\d+\s*', line.strip())).strip() # Filter out lines that are just numbers
        else:
             content = "" # Handle cases with only one line (the ID line)


        # Simple heuristic to avoid matching things that are not proper MS entries
        # Added check for minimum length after stripping potential marks/numbers
        if len(content.split()) < 3 and "mark" not in content.lower() and not re.search(r'[a-zA-Z0-9]', content): # Added check for alphanumeric content
            logging.debug(f"Skipping potential MS entry due to heuristic: ID='{full_id_raw}', Content='{content[:50]}...'") # Changed to debug
            continue

        new_entry = {
            "question_id": full_id,
            "text": content,
            "page_number": 0, # Placeholder, will be corrected later based on match position if possible
            "bbox": (0, 0, 0, 0), # Placeholder
            "diagrams": [] # Placeholder for diagrams
        }
        # TODO: Add logic to find the actual page number and bbox for the matched ID
        marking_schemes.append(new_entry)

    if not marking_schemes:
        logging.warning(f"No structured marking scheme entries found in {pdf_path} based on pattern.")


    return marking_schemes

def assign_diagrams_to_marking_schemes(marking_schemes, diagrams):
    """
    Assigns diagrams to the closest marking scheme entry based on vertical proximity
    and page number. Note: This relies on accurate page/bbox info for MS entries,
    which is currently a placeholder.
    """
    if not marking_schemes or not diagrams:
        return marking_schemes

    logging.info(f"Attempting to assign {len(diagrams)} diagrams to {len(marking_schemes)} marking scheme entries.")

    # Currently, MS entries have placeholder page/bbox.
    # Diagram assignment based on vertical proximity is not meaningful without accurate locations.
    # If accurate MS locations are added, this logic would need to be updated to use them.

    # For now, we'll just return the marking schemes as is, the diagram list might still be useful globally.
    # If diagrams are truly associated per question part in the MS, the extraction logic needs refinement.

    # If diagrams need to be assigned, the logic here would iterate through diagrams,
    # find the closest MS entry (on the same page if page numbers are accurate),
    # and append the diagram info to the MS entry's 'diagrams' list.

    # Example (if MS entries had accurate 'bbox' and 'page_number'):
    # for diagram in diagrams:
    #     min_dist = float('inf')
    #     best_match = None
    #     diagram_center_y = (diagram['bbox'][1] + diagram['bbox'][3]) / 2
    #
    #     for ms in marking_schemes:
    #         # Only consider entries on the same page
    #         if 'page_number' in ms and ms['page_number'] == diagram['page_number']:
    #             if 'bbox' in ms and len(ms['bbox']) == 4:
    #                 ms_center_y = (ms['bbox'][1] + ms['bbox'][3]) / 2
    #                 dist = _dist_vertical(diagram_center_y, ms_center_y)
    #                 if dist < min_dist:
    #                     min_dist = dist
    #                     best_match = ms
    #
    #     if best_match:
    #         if 'diagrams' not in best_match:
    #             best_match['diagrams'] = []
    #         best_match["diagrams"].append(diagram)
    #     else:
    #         logging.warning(f"Diagram {diagram['diagram_id']} could not be assigned to any marking scheme entry.")


    return marking_schemes # Return the original marking schemes list


def process_marking_scheme_document(pdf_path, diagram_out_dir, json_out_path):
    """Main function to process a single marking scheme document."""
    logging.info(f"Processing Marking Scheme PDF: {pdf_path}") # Added logging
    # Ensure output directory for JSON exists
    json_output_dir = os.path.dirname(json_out_path)
    if not json_output_dir: # Handle case where json_out_path is just a filename
        json_output_dir = '.'
    try:
        os.makedirs(json_output_dir, exist_ok=True)
        logging.info(f"Ensured output directory for JSON exists: {json_output_dir}")
    except Exception as e:
         logging.error(f"Error creating output directory for JSON {json_output_dir}: {e}", exc_info=True)
         # If cannot create output directory, cannot save, so return/exit
         return # Or sys.exit(1) if this is fatal

    # 1) Extract structured marking scheme data
    logging.info("⋯ extracting structured marking scheme data...") # Added logging
    marking_schemes = extract_marking_scheme_data(pdf_path)

    # 2) Extract all diagrams
    logging.info("⋯ extracting diagrams and images…") # Added logging
    # Note: The current extract_diagrams in this script saves full pages, not cropped images, and doesn't use OCR.
    # If cropped diagrams with captions are needed, this function should be updated.
    diagrams = extract_diagrams(pdf_path, diagram_out_dir)

    # 3) Assign diagrams to the marking schemes
    logging.info("⋯ assigning diagrams to marking scheme entries…") # Added logging
    # Note: Diagram assignment is currently limited as MS entries lack accurate location data.
    enriched_marking_schemes = assign_diagrams_to_marking_schemes(marking_schemes, diagrams)

    # 4) Save the final JSON
    if enriched_marking_schemes: # Only attempt to save if there are entries
        try:
            with open(json_out_path, "w", encoding="utf-8") as f:
                json.dump(enriched_marking_schemes, f, indent=2)
            logging.info(f"✓ Saved extracted marking scheme data to {json_out_path}") # Added logging
            print(f"✓ Saved extracted marking scheme data to {json_out_path}") # Added print for immediate feedback
        except Exception as e: # Add try-except around JSON saving
            logging.error(f"Error saving JSON to {json_out_path}: {e}", exc_info=True)
            print(f"Error saving JSON to {json_out_path}: {e}") # Added print for immediate feedback
            # Decide how to handle save error - return or sys.exit(1)
            # For pipeline, probably want to log and continue if possible, or let main handle it.
            # For now, log and don't exit.
    else:
        logging.warning(f"No marking scheme entries to save for {pdf_path}.")
        print(f"No marking scheme entries to save for {pdf_path}.") # Added print for immediate feedback


if __name__ == "__main__":
    # Add try-except around the main call for robust execution via subprocess
    try:
        logging.info("Starting marking_scheme_to_text_js.py execution...") # Added logging
        if len(sys.argv) != 4:
            logging.error("Usage: python marking_scheme_to_text_js.py <pdf_path> <diagram_out_dir> <json_out_path>") # Changed print to logging
            print("Usage: python marking_scheme_to_text_js.py <pdf_path> <diagram_out_dir> <json_out_path>") # Added print for immediate feedback
            sys.exit(1) # Exit with status 1 if usage is incorrect

        pdf_path = sys.argv[1]
        diagram_out_dir = sys.argv[2]
        json_out_path = sys.argv[3]

        process_marking_scheme_document(pdf_path, diagram_out_dir, json_out_path)
        logging.info("marking_scheme_to_text_js.py finished successfully.") # Added logging

    except Exception as e: # Catch any unhandled exceptions
        logging.error(f"An unhandled error occurred during marking_scheme_to_text_js.py execution: {e}", exc_info=True)
        sys.exit(1) # Ensure script exits with error status on unhandled exceptions

In [ ]:
!pip install pytesseract